# DDRI Representative-Station Static Weather Interaction Comparison(정적 피처-날씨 상호작용 비교)

대표 대여소 15개 `second_round_data(2차 실험용 통합 피처모음)` 기준으로 정적 피처(static feature, 정적 입력 변수)와 weather interaction(날씨 상호작용 피처) 조합이 실제로 개선되는지 확인한다.

비교 대상:
- `rep15_static_base(대표 15개 정적 피처 기준선 모델)`
- `rep15_static_weather_full(대표 15개 정적 피처 + 전체 날씨 상호작용 조합)`


In [1]:
from pathlib import Path

import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ROOT = Path('/Users/cheng80/Desktop/ddri_work')
DATA_DIR = ROOT / '3조 공유폴더/대표대여소_예측데이터_15개/second_round_data'
OUTPUT_DATA_DIR = ROOT / 'works/05_prediction_long/output/data'
OUTPUT_DATA_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_PATH = DATA_DIR / 'ddri_prediction_long_train_2023_2024_second_round_feature_collection.csv'
TEST_PATH = DATA_DIR / 'ddri_prediction_long_test_2025_second_round_feature_collection.csv'

In [2]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

for df in [train, test]:
    df['date'] = pd.to_datetime(df['date'])
    df['datetime'] = df['date'] + pd.to_timedelta(df['hour'].astype('int16'), unit='h')

train.shape, test.shape

((263160, 52), (131400, 52))

In [3]:
def add_weather_interactions(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy()
    result['rain_x_commute'] = (result['is_rainy'] * result['is_commute_hour']).astype('int8')
    result['rain_x_morning_commute'] = (result['is_rainy'] * result['hour'].isin([7, 8, 9]).astype('int8')).astype('int8')
    result['rain_x_evening_commute'] = (result['is_rainy'] * result['hour'].isin([17, 18, 19]).astype('int8')).astype('int8')
    result['rain_x_night'] = (result['is_rainy'] * result['is_night_hour']).astype('int8')
    result['rain_x_lunch'] = (result['is_rainy'] * result['is_lunch_hour']).astype('int8')
    result['heavy_rain_x_commute'] = (result['heavy_rain_flag'] * result['is_commute_hour']).astype('int8')
    result['heavy_rain_x_night'] = (result['heavy_rain_flag'] * result['is_night_hour']).astype('int8')
    result['precipitation_x_commute'] = (result['precipitation'] * result['is_commute_hour']).astype('float32')
    result['precipitation_x_night'] = (result['precipitation'] * result['is_night_hour']).astype('float32')
    result['precipitation_x_lunch'] = (result['precipitation'] * result['is_lunch_hour']).astype('float32')
    return result

train_feat = add_weather_interactions(train)
test_feat = add_weather_interactions(test)
train_feat.shape, test_feat.shape

((263160, 62), (131400, 62))

In [4]:
BASE_CATEGORICAL = ['station_id', 'station_group', 'cluster', 'mapped_dong_code', 'hour', 'weekday', 'month', 'holiday']
STATIC_BASE_FEATURES = [
    'temperature', 'humidity', 'precipitation', 'wind_speed',
    'subway_distance_m', 'bus_stop_count_300m', 'station_elevation_m', 'elevation_diff_nearest_subway_m',
    'nearest_park_area_sqm', 'distance_naturepark_m', 'distance_river_boundary_m',
    'restaurant_count_300m', 'cafe_count_300m', 'convenience_store_count_300m', 'pharmacy_count_300m',
    'food_retail_count_1000m', 'fitness_count_500m', 'cinema_count_1000m',
    'log1p_restaurant_count_300m', 'log1p_cafe_count_300m', 'log1p_convenience_store_count_300m',
    'log1p_pharmacy_count_300m', 'log1p_food_retail_count_1000m', 'log1p_fitness_count_500m', 'log1p_cinema_count_1000m',
    'is_weekend', 'is_commute_hour', 'is_lunch_hour', 'is_night_hour', 'is_rainy', 'heavy_rain_flag',
    'hour_sin', 'hour_cos', 'is_holiday_eve', 'is_after_holiday', 'lag_48h', 'rolling_mean_6h', 'rolling_std_6h'
]
WEATHER_FULL = [
    'rain_x_commute', 'rain_x_morning_commute', 'rain_x_evening_commute', 'rain_x_night', 'rain_x_lunch',
    'heavy_rain_x_commute', 'heavy_rain_x_night', 'precipitation_x_commute', 'precipitation_x_night', 'precipitation_x_lunch'
]
BASE_FEATURE_COLS = BASE_CATEGORICAL + STATIC_BASE_FEATURES
WEATHER_FEATURE_COLS = BASE_FEATURE_COLS + WEATHER_FULL

pd.DataFrame({
    'feature_set': ['rep15_static_base', 'rep15_static_weather_full'],
    'feature_count': [len(BASE_FEATURE_COLS), len(WEATHER_FEATURE_COLS)]
})

,feature_set,feature_count
0,rep15_static_base,46
1,rep15_static_weather_full,56


In [5]:
def evaluate_predictions(model_name: str, split_name: str, y_true: pd.Series, y_pred) -> dict:
    return {
        'model': model_name,
        'split': split_name,
        'rmse': round(float(mean_squared_error(y_true, y_pred) ** 0.5), 4),
        'mae': round(float(mean_absolute_error(y_true, y_pred)), 4),
        'r2': round(float(r2_score(y_true, y_pred)), 4),
    }


def build_lgbm_model():
    return LGBMRegressor(
        objective='regression',
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
    )


def prepare_xy(df: pd.DataFrame, feature_cols: list[str]):
    X = df[feature_cols].copy()
    for col in [c for c in feature_cols if c in BASE_CATEGORICAL]:
        X[col] = X[col].astype('category')
    y = df['rental_count'].copy()
    return X, y

In [6]:
train_2023 = train_feat[train_feat['date'] < '2024-01-01'].copy()
valid_2024 = train_feat[train_feat['date'] >= '2024-01-01'].copy()
full_train = train_feat.copy()

results = []

for model_name, feature_cols in [
    ('rep15_static_base', BASE_FEATURE_COLS),
    ('rep15_static_weather_full', WEATHER_FEATURE_COLS),
]:
    X_train, y_train = prepare_xy(train_2023, feature_cols)
    X_valid, y_valid = prepare_xy(valid_2024, feature_cols)
    X_full, y_full = prepare_xy(full_train, feature_cols)
    X_test, y_test = prepare_xy(test_feat, feature_cols)

    model = build_lgbm_model()
    cat_cols = [c for c in feature_cols if c in BASE_CATEGORICAL]
    model.fit(X_train, y_train, categorical_feature=cat_cols)
    valid_pred = model.predict(X_valid)
    results.append(evaluate_predictions(model_name, 'validation_2024', y_valid, valid_pred))

    model.fit(X_full, y_full, categorical_feature=cat_cols)
    test_pred = model.predict(X_test)
    results.append(evaluate_predictions(model_name, 'test_2025_refit', y_test, test_pred))

results_df = pd.DataFrame(results)
results_df

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003354 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1431
[LightGBM] [Info] Number of data points in the train set: 131400, number of used features: 46
[LightGBM] [Info] Start training from score 0.722405


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005372 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1460
[LightGBM] [Info] Number of data points in the train set: 263160, number of used features: 46
[LightGBM] [Info] Start training from score 0.724514


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004019 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1607
[LightGBM] [Info] Number of data points in the train set: 131400, number of used features: 56
[LightGBM] [Info] Start training from score 0.722405


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007158 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1670
[LightGBM] [Info] Number of data points in the train set: 263160, number of used features: 56
[LightGBM] [Info] Start training from score 0.724514


,model,split,rmse,mae,r2
0,rep15_static_base,validation_2024,1.0173,0.6190,0.5612
1,rep15_static_base,test_2025_refit,0.9196,0.5689,0.5339
2,rep15_static_weather_full,validation_2024,1.0188,0.6194,0.5599
3,rep15_static_weather_full,test_2025_refit,0.9198,0.5693,0.5336


In [7]:
metrics_path = OUTPUT_DATA_DIR / 'ddri_rep15_static_weather_interaction_metrics.csv'
results_df.to_csv(metrics_path, index=False, encoding='utf-8-sig')
print('saved:', metrics_path)
results_df

saved: /Users/cheng80/Desktop/ddri_work/works/05_prediction_long/output/data/ddri_rep15_static_weather_interaction_metrics.csv


,model,split,rmse,mae,r2
0,rep15_static_base,validation_2024,1.0173,0.6190,0.5612
1,rep15_static_base,test_2025_refit,0.9196,0.5689,0.5339
2,rep15_static_weather_full,validation_2024,1.0188,0.6194,0.5599
3,rep15_static_weather_full,test_2025_refit,0.9198,0.5693,0.5336
